# 06_mllm / gemini_setup.ipynb

**Purpose:** One-time setup for the Gemini 2.5 Flash multimodal LLM used to generate human-readable explanations of GSAF predictions.

**What this notebook does:**
1. Installs `google-generativeai` if missing.
2. Loads your Gemini API key from `.env` or prompts you to paste it.
3. Verifies the API key with a text-only call.
4. Verifies vision works with a single image-caption pair from the dataset.
5. Saves a small helper module (`src/utils/gemini_client.py`) that the next two notebooks (`mllm_inference.ipynb`, `chain_of_thought.ipynb`) will import.
6. Sets up a disk-backed cache so repeat calls do not re-bill the API.

**Free-tier reminder (Gemini 2.5 Flash, as of April 2026):**
- ~10 requests / minute
- ~250 requests / day
- The next two notebooks will run on a stratified 500-pair sample; that takes ~2 calendar days at free-tier limits.

**Inputs:**
- A Gemini API key from https://aistudio.google.com/apikey
- Stored in `.env` at repo root as `GEMINI_API_KEY=...`
- Optional: one local image from `data/processed/ooc_pairs/images/` for the smoke test

**Outputs:**
- `src/utils/gemini_client.py` — the shared client + cache module
- `cache/mllm/.gemini_setup_ok` — sentinel file so future notebooks can verify setup ran


In [1]:
# 1. Install dependency (idempotent)
import subprocess, sys
try:
    import openai
    print("openai already installed:", openai.__version__)
except ImportError:
    print("Installing openai…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "Pillow"])
    import openai
    print("Installed.")

openai already installed: 2.32.0


In [2]:
# 2. Path setup
from __future__ import annotations
import os, sys, json, hashlib, time
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import ALL_MANIFEST, PROCESSED_IMAGES, VERITE_IMAGES

CACHE_MLLM = ROOT / "cache" / "mllm"
CACHE_MLLM.mkdir(parents=True, exist_ok=True)
GEMINI_CACHE = CACHE_MLLM / "gemini_responses"
GEMINI_CACHE.mkdir(parents=True, exist_ok=True)

ENV_PATH = ROOT / ".env"
print("Repo root:", ROOT)
print("MLLM cache:", CACHE_MLLM)
print(".env exists:", ENV_PATH.exists())


Repo root: e:\eve-rag
MLLM cache: e:\eve-rag\cache\mllm
.env exists: True


## 3. Load API key

If you don't have one yet:
1. Visit https://aistudio.google.com/apikey
2. Click "Create API key"
3. Add this line to `E:\\eve-rag\\.env`:
   ```
   GEMINI_API_KEY=AIzaSy…
   ```
4. Re-run this cell.


In [3]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=ENV_PATH, override=True)

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Paste your OPENROUTER_API_KEY (won't be saved): ").strip()

assert OPENROUTER_API_KEY, "No API key provided. Get one free at https://openrouter.ai"
print(f"API key loaded (length={len(OPENROUTER_API_KEY)}, prefix={OPENROUTER_API_KEY[:6]}…)")

API key loaded (length=73, prefix=sk-or-…)


## 4. Configure the SDK and pick a model

In [4]:
# Try models in order — first available wins
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

CANDIDATE_MODELS = [
    "google/gemma-4-31b-it:free",       # best free vision on OR right now
    "google/gemma-4-26b-a4b-it:free",   # smaller gemma 4 variant
    "meta-llama/llama-3.2-11b-vision-instruct:free",  # fallback
    "mistralai/mistral-small-3.1-24b-instruct:free",  # fallback (text+vision)
]

MODEL_NAME = None
for cand in CANDIDATE_MODELS:
    try:
        test = client.chat.completions.create(
            model=cand,
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=5,
        )
        MODEL_NAME = cand
        print(f"✓ Available: {cand}")
        break
    except Exception as e:
        print(f"✗ Unavailable: {cand}  ({str(e)[:80]})")

assert MODEL_NAME is not None, "No free vision model available on OpenRouter right now."
print(f"\nUsing: {MODEL_NAME}")

✗ Unavailable: google/gemma-4-31b-it:free  (Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, )
✓ Available: google/gemma-4-26b-a4b-it:free

Using: google/gemma-4-26b-a4b-it:free


## 5. Smoke test 1 — text only

In [5]:
# Smoke test 1 — text only
resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Reply with exactly: 'EVE-RAG setup OK'"}],
    max_tokens=20,
)
reply = resp.choices[0].message.content.strip()
print("Text reply:", reply)
assert "EVE-RAG" in reply or "OK" in reply, "Unexpected reply"

Text reply: EVE-RAG setup OK


## 6. Smoke test 2 — image + caption

In [6]:
# Smoke test 2 — image + caption
import base64, pandas as pd
from PIL import Image
from io import BytesIO

m = pd.read_csv(ALL_MANIFEST)
candidates = m[(m["has_image"] == 1) & (m["dataset"].isin(["mmfakebench", "verite"]))].head(50)

test_row = None
test_img = None
for _, r in candidates.iterrows():
    p = Path(str(r.get("image_path") or ""))
    if not p.is_absolute():
        for base in [PROCESSED_IMAGES, VERITE_IMAGES, ROOT]:
            cand = base / p.name
            if cand.exists(): p = cand; break
            cand2 = base / p
            if cand2.exists(): p = cand2; break
    if p.exists():
        test_row = r; test_img = p; break

assert test_row is not None, "No local image found"
print("Test image:", test_img)
print("Caption   :", test_row["caption"][:140])

img = Image.open(test_img).convert("RGB")
img.thumbnail((512, 512))
buf = BytesIO()
img.save(buf, format="JPEG")
b64 = base64.b64encode(buf.getvalue()).decode()

prompt = (
    "You are an analyst checking whether an image and a caption match. "
    "In one sentence, describe the image. "
    "In one more sentence, say whether it plausibly matches the caption: "
    f"'{test_row['caption']}'."
)

resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
    ]}],
    max_tokens=200,
)
print("\nReply:", resp.choices[0].message.content.strip())

C:\Users\User\AppData\Local\Temp\ipykernel_9764\239511774.py:6: DtypeWarning: Columns (0: image_path, 1: source_image_id, 2: text_source, 3: image_source, 4: fake_cls, 5: raw_label) have mixed types. Specify dtype option on import or set low_memory=False.
  m = pd.read_csv(ALL_MANIFEST)


Test image: E:\eve-rag\data\processed\ooc_pairs\images\verite_true_0.jpg
Caption   : Photograph shows a family that was taken away by local epidemic prevention personnel after they had returned home from Hubei Province in Feb


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-26b-a4b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_2z57tN6xjOabJMqU2dMOvIlrPQH'}

## 7. Build the shared helper module

In [ ]:
HELPER_PATH = ROOT / "src" / "utils" / "mllm_client.py"

helper_code = '''"""
Shared OpenRouter/Qwen2.5-VL client for EVE-RAG.

Usage:
    from src.utils.mllm_client import MLLMClient
    gc = MLLMClient()
    text = gc.generate(prompt, image_path=Path("x.jpg"))
"""
from __future__ import annotations
import os, json, time, hashlib, base64
from pathlib import Path
from dataclasses import dataclass, field
from io import BytesIO

from openai import OpenAI
from PIL import Image
from dotenv import load_dotenv


@dataclass
class MLLMClient:
    model_name: str = "qwen/qwen2.5-vl-72b-instruct:free"
    rpm_limit: int = 18
    cache_dir: Path = None
    image_max_side: int = 512
    max_retries: int = 5
    request_log: list = field(default_factory=list)

    def __post_init__(self):
        here = Path(__file__).resolve()
        for p in [here] + list(here.parents):
            if (p / "src" / "utils" / "paths.py").exists():
                self.repo_root = p; break
        else:
            self.repo_root = Path.cwd()
        load_dotenv(self.repo_root / ".env", override=True)
        api_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
        if not api_key:
            raise RuntimeError("OPENROUTER_API_KEY not in .env")
        self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
        if self.cache_dir is None:
            self.cache_dir = self.repo_root / "cache" / "mllm" / "openrouter_responses"
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _key(prompt: str, image_path, model_name: str) -> str:
        h = hashlib.sha256()
        h.update(model_name.encode())
        h.update(b"||")
        h.update(prompt.encode("utf-8"))
        h.update(b"||")
        if image_path is not None and Path(image_path).exists():
            h.update(Path(image_path).read_bytes())
        return h.hexdigest()[:32]

    def _wait_if_needed(self):
        now = time.time()
        self.request_log = [t for t in self.request_log if now - t < 60]
        if len(self.request_log) >= self.rpm_limit:
            sleep_for = 60 - (now - self.request_log[0]) + 0.5
            if sleep_for > 0:
                time.sleep(sleep_for)
                self.request_log = [t for t in self.request_log if time.time() - t < 60]

    def generate(self, prompt: str, image_path=None,
                 use_cache: bool = True, temperature: float = 0.2,
                 max_output_tokens: int = 512) -> str:
        key = self._key(prompt, image_path, self.model_name)
        cache_file = self.cache_dir / f"{key}.json"
        if use_cache and cache_file.exists():
            try:
                return json.loads(cache_file.read_text(encoding="utf-8"))["text"]
            except Exception:
                pass

        self._wait_if_needed()
        content = [{"type": "text", "text": prompt}]
        if image_path is not None:
            img = Image.open(image_path).convert("RGB")
            img.thumbnail((self.image_max_side, self.image_max_side))
            buf = BytesIO()
            img.save(buf, format="JPEG")
            b64 = base64.b64encode(buf.getvalue()).decode()
            content.append({"type": "image_url",
                             "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})

        last_err = None
        for attempt in range(self.max_retries):
            try:
                self.request_log.append(time.time())
                resp = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": content}],
                    temperature=temperature,
                    max_tokens=max_output_tokens,
                )
                text = (resp.choices[0].message.content or "").strip()
                cache_file.write_text(json.dumps({
                    "text": text, "prompt": prompt[:2000],
                    "image": str(image_path) if image_path else None,
                    "model": self.model_name, "ts": time.time(),
                }), encoding="utf-8")
                return text
            except Exception as e:
                last_err = e
                msg = str(e).lower()
                if any(k in msg for k in ("429", "quota", "rate", "502", "503", "500")):
                    backoff = min(60.0, 2 ** attempt * 5)
                    time.sleep(backoff)
                    continue
                raise
        raise RuntimeError(f"MLLMClient failed after {self.max_retries} retries: {last_err}")
'''

HELPER_PATH.write_text(helper_code, encoding="utf-8")
print("Wrote helper module:", HELPER_PATH)

Wrote helper module: e:\eve-rag\src\utils\gemini_client.py
(4,715 chars)


## 8. Verify helper module works end-to-end

In [ ]:
import importlib, sys
if "src.utils.mllm_client" in sys.modules:
    importlib.reload(sys.modules["src.utils.mllm_client"])
from src.utils.mllm_client import MLLMClient

gc = MLLMClient()
print("Helper loaded. Model =", gc.model_name)

out = gc.generate(
    prompt="In one sentence, what does this image show?",
    image_path=test_img,
    use_cache=True,
    temperature=0.0,
    max_output_tokens=120,
)
print("\nHelper test reply:")
print(out)

Helper loaded. Model = models/gemini-2.5-flash

Helper test reply:
The image shows a


## 9. Write sentinel file

In [ ]:
sentinel = CACHE_MLLM / ".mllm_setup_ok"
sentinel.write_text(json.dumps({
    "model_name": MODEL_NAME,
    "provider": "openrouter",
    "ts": time.time(),
    "rpm_limit": 18,
}, indent=2), encoding="utf-8")
print("Wrote sentinel:", sentinel)
print()
print("✓ Setup complete. Proceed to mllm_inference.ipynb")

Wrote sentinel: e:\eve-rag\cache\mllm\.gemini_setup_ok

✓ Setup complete. Proceed to 06_mllm/mllm_inference.ipynb


---
**Done.** The next notebook (`mllm_inference.ipynb`) imports `from src.utils.mllm_client import MLLMClient` and uses it for batch explanation generation.